# Weighted Deduplication Workflow

> Example for running the deduplication pipeline on a reference CSV.

Note: This example assumes you have a dataset identifier (`recordid`) and uses the PaperWithId data model.  

This notebook includes the following steps to deduplicate references from a CSV: 
1. Load the original CSV into a DataFrame so dataset identifiers are retained.
2. Load and validate the bibliographic fields as `Paper` objects.
3. Use positional indices (`index_a`, `index_b`) while blocking and scoring.
4. Map those positions back to `recordid` values for review and record resolution.
5. Resolve predicted duplicate clusters (retain unique records) and export the results.

## 1. Imports and configuration

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from loguru import logger

REPO_DIR = Path.cwd()
if not (REPO_DIR / "app").is_dir() and (REPO_DIR.parent / "app").is_dir():
    REPO_DIR = REPO_DIR.parent

if not (REPO_DIR / "app").is_dir():
    raise RuntimeError(
        "Could not locate the repository root. "
        "Open this notebook from the project root or notebooks directory."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from app.candidate_selection import build_blocked_pairs
from app.data_models import PaperWithId
from app.dedupe import Deduper
from app.import_references import DEFAULT_COLUMNS, CsvLoadConfig, load_reference_csv
from app.record_resolution import (
    RecordResolutionConfig,
    records_to_dataframe,
    resolve_records,
)

INPUT_CSV = REPO_DIR / "notebooks/data/diabetes_data.csv"
OUTPUT_DIR = REPO_DIR / "notebooks/data"

THRESHOLD = 0.85
RETENTION_STRATEGY = "prefer_doi_abstract"
ENRICH_KEPT_RECORDS = True

logger.remove()
logger.add(sys.stderr, level="INFO")

ImportError: cannot import name 'RecordResolutionConfig' from 'app.record_resolution' (c:\Coding_projects\deduplication-toolkit\app\record_resolution.py)

## 2. Load validated records once

This cell assumes `CsvLoadConfig` has an `include_record_id` option and that
`load_reference_csv()` returns `PaperWithId` objects when it is enabled.

The loader remains responsible for CSV encoding, column normalization, parsing, and
Pydantic validation. The notebook does not call `pandas.read_csv()` separately.

In [ ]:
input_columns = list(
    dict.fromkeys(
        [
            *DEFAULT_COLUMNS,
            "recordid",
        ]
    )
)

papers = load_reference_csv(
    INPUT_CSV,
    CsvLoadConfig(
        columns=input_columns,
        include_record_id=True,
    ),
)

if not papers:
    raise ValueError(
        f"No valid papers were loaded from {INPUT_CSV}"
    )

unexpected_types = sorted(
    {
        type(paper).__name__
        for paper in papers
        if not isinstance(paper, PaperWithId)
    }
)
if unexpected_types:
    raise TypeError(
        "Expected load_reference_csv() to return PaperWithId objects. "
        f"Unexpected types: {unexpected_types}. "
        "Apply the include_record_id loader patch first."
    )

record_ids = [
    paper.recordid
    for paper in papers
]
if len(record_ids) != len(set(record_ids)):
    raise ValueError("PaperWithId.recordid values must be unique.")

print(f"Loaded {len(papers):,} validated PaperWithId records")
display(records_to_dataframe(papers).head())

Loaded 1,845 validated PaperWithId records


,doi,openalex_id,pubmed_id,isbn,issn,title,authors,year,journal,publisher,pages,volume,issue,abstract,recordid
0,"{'identifier': '10.1097/mol.0b013e3282ffaf82',...",None,None,None,None,Intestinal lipoprotein overproduction in insul...,"[{'display_name': 'Adeli K', 'orcid': None, 'p...",2008,Curr Opin Lipidol,None,221-8,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,1
1,"{'identifier': '10.1097/mol.0b013e3282ffaf82',...",None,None,None,None,Intestinal lipoprotein overproduction in insul...,"[{'display_name': 'Adeli K', 'orcid': None, 'p...",2008,Curr Opin Lipidol,None,221-228,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,699
2,"{'identifier': '10.1097/mol.0b013e3282ffaf82',...",None,None,None,None,Intestinal lipoprotein overproduction in insul...,"[{'display_name': 'Adeli K', 'orcid': None, 'p...",2008,Curr Opin Lipidol,None,221-8,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,728
3,"{'identifier': '10.1097/mol.0b013e3282ffaf82',...",None,None,None,None,Intestinal lipoprotein overproduction in insul...,"[{'display_name': 'Adeli K', 'orcid': None, 'p...",2008,Curr Opin Lipidol,None,221-228,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,1447
4,None,None,None,None,None,SGLT2 inhibition improves coronary microvascul...,"[{'display_name': 'Adingupu D. D', 'orcid': No...",2017,European Heart Journal,None,893,38,None,Background: Trea]t with SGLT2i have been sugge...,1630


## 3. Generate candidate pairs

`build_blocked_pairs()` works with the bibliographic fields inherited from `Paper`.
It returns positions in the supplied sequence as `index_a` and `index_b`.

The source IDs are added only for readable inspection and export.

In [ ]:
pairs_df = build_blocked_pairs(
    papers,
    include_block_rules=True,
)

if pairs_df.empty:
    raise ValueError("Blocking produced no candidate pairs.")

required_pair_columns = {"index_a", "index_b"}
missing_pair_columns = required_pair_columns.difference(pairs_df.columns)
if missing_pair_columns:
    raise ValueError(
        "build_blocked_pairs() must return positional pair columns. "
        f"Missing: {sorted(missing_pair_columns)}. "
        f"Available: {pairs_df.columns.tolist()}"
    )

pairs_df = pairs_df.copy()
pairs_df["index_a"] = pairs_df["index_a"].astype(int)
pairs_df["index_b"] = pairs_df["index_b"].astype(int)

pairs_df["id_a"] = [
    papers[index].recordid
    for index in pairs_df["index_a"]
]
pairs_df["id_b"] = [
    papers[index].recordid
    for index in pairs_df["index_b"]
]

front_columns = ["index_a", "index_b", "id_a", "id_b"]
pairs_df = pairs_df[
    front_columns
    + [
        column
        for column in pairs_df.columns
        if column not in front_columns
    ]
]

print(f"Generated {len(pairs_df):,} candidate pairs")
display(pairs_df.head())

Generated 8,487 candidate pairs


,index_a,index_b,id_a,id_b,block_rules
0,0,1,1,699,"doi | pages,volume | title | year,journal | ye..."
1,0,2,1,728,"abstract | doi | pages,volume | title | year,j..."
2,0,3,1,1447,"doi | pages,volume | title | year,journal | ye..."
3,1,2,699,728,"doi | pages,volume | title | year,journal | ye..."
4,1,3,699,1447,"abstract | doi | pages,volume | title | year,j..."


## 4. Initialise and run weighted scoring

In [ ]:
first_pair = pairs_df.iloc[0]

deduper = Deduper(
    reference=papers[int(first_pair["index_a"])],
    candidates=[papers[int(first_pair["index_b"])]],
)

probabilities = []

for row in pairs_df.itertuples(index=False):
    probability = deduper.dedupe_weighted(
        papers[int(row.index_a)],
        papers[int(row.index_b)],
    )
    probabilities.append(float(probability))

scored_pairs_df = pairs_df.copy()
scored_pairs_df["probability"] = probabilities
scored_pairs_df["duplicate_prediction"] = (
    scored_pairs_df["probability"] >= THRESHOLD
)

print(
    f"Predicted {scored_pairs_df['duplicate_prediction'].sum():,} "
    f"duplicate pairs at threshold {THRESHOLD:.2f}"
)
display(
    scored_pairs_df.sort_values(
        "probability",
        ascending=False,
    ).head(20)
)

2026-07-30 18:40:26.263 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.264 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.266 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.266 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.267 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.268 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.269 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.269 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.270 | INFO     | app.dedupe:should_early_stop:481 - No early stopping reason detected.
2026-07-30 18:40:26.271 | INFO     | 

Predicted 2,531 duplicate pairs at threshold 0.85


,index_a,index_b,id_a,id_b,block_rules,probability,duplicate_prediction
0,0,1,1,699,"doi | pages,volume | title | year,journal | ye...",0.999997,True
1,0,2,1,728,"abstract | doi | pages,volume | title | year,j...",0.999997,True
2,0,3,1,1447,"doi | pages,volume | title | year,journal | ye...",0.999997,True
3,1,2,699,728,"doi | pages,volume | title | year,journal | ye...",0.999997,True
4,1,3,699,1447,"abstract | doi | pages,volume | title | year,j...",0.999997,True
5,2,3,728,1447,"doi | pages,volume | title | year,journal | ye...",0.999997,True
2061,1495,1497,417,1158,"doi | pages,volume | title | year,journal | ye...",0.999997,True
832,560,566,1819,1152,"doi | pages,volume | title | year,journal | ye...",0.999997,True
833,561,562,412,69,"doi | pages,volume | title | year,journal | ye...",0.999997,True
834,561,565,412,802,"doi | pages,volume | title | year,journal | ye...",0.999997,True


## 5. Resolve duplicate clusters

`resolve_records()` consumes the same `PaperWithId` sequence used for blocking and
scoring. It clusters predicted duplicate edges by `index_a` and `index_b`, chooses a
canonical object, and optionally enriches missing fields from validated cluster
peers.

In [ ]:
resolution = resolve_records(
    records=papers,
    scored_pairs=scored_pairs_df,
    config=RecordResolutionConfig(
        threshold=THRESHOLD,
        strategy=RETENTION_STRATEGY,
        enrich_kept_records=ENRICH_KEPT_RECORDS,
    ),
)

deduplicated_df = records_to_dataframe(
    resolution.kept_records
)
removed_duplicates_df = records_to_dataframe(
    resolution.removed_records
)
decisions_df = resolution.decisions_df

duplicate_cluster_rows = decisions_df.loc[
    decisions_df["cluster_size"] > 1
]

print(f"Original records: {len(papers):,}")
print(f"Deduplicated records kept: {len(resolution.kept_records):,}")
print(f"Records removed as duplicates: {len(resolution.removed_records):,}")
print(
    "Records involved in predicted duplicate clusters: "
    f"{len(duplicate_cluster_rows):,}"
)

display(
    duplicate_cluster_rows[
        [
            "index",
            "recordid",
            "predicted_cluster",
            "cluster_size",
            "keep",
            "kept_recordid",
        ]
    ].head(20)
)
display(deduplicated_df.head())

Original records: 1,845
Deduplicated records kept: 593
Records removed as duplicates: 1,252
Records involved in predicted duplicate clusters: 1,816


,index,recordid,predicted_cluster,cluster_size,keep,kept_recordid
0,1060,145,344,8,False,488
1,1089,469,344,8,False,488
2,1091,491,344,8,False,488
3,1757,493,98,8,False,494
4,1092,883,344,8,False,488
5,1058,1212,344,8,False,488
6,1059,1232,344,8,False,488
7,1093,1235,344,8,False,488
8,1758,1237,98,8,False,494
9,275,1238,98,8,False,494


,doi,openalex_id,pubmed_id,isbn,issn,title,authors,year,journal,publisher,pages,volume,issue,abstract,recordid
0,"{'identifier': '10.1002/phar.1547', 'identifie...",None,None,None,None,Vascular protection with dipeptidyl peptidase-...,"[{'display_name': 'Ahmed H. A', 'orcid': None,...",2015,Pharmacotherapy,None,277-97,35,3,The dipeptidyl peptidase-IV (DPP-IV) inhibitor...,2
1,"{'identifier': '10.1016/j.vph.2017.07.001', 'i...",None,None,None,None,Dipeptidyl peptidase IV inhibitors as novel re...,"[{'display_name': 'Akoumianakis I', 'orcid': N...",2017,Vascul Pharmacol,None,01-Apr,96-98,None,Dipeptidyl peptidase IV (DPP-IV) has been reve...,4
2,"{'identifier': '10.3389/fendo.2012.00112', 'id...",None,None,None,None,Incretin hormones as immunomodulators of ather...,"[{'display_name': 'Alonso N', 'orcid': None, '...",2012,Front Endocrinol (Lausanne),None,112,3,None,Atherosclerosis results from endothelial cell ...,6
3,None,None,None,None,None,[Comparative characteristics of in vivo models...,"[{'display_name': 'Apryatin S. A', 'orcid': No...",2016,Vopr Pitan,None,14-23,85,6,In vivo simulation of lipid disorders (hyperli...,9
4,"{'identifier': '10.1007/s11892-018-1043-z', 'i...",None,None,None,None,Cardiovascular Effects of Different GLP-1 Rece...,"[{'display_name': 'Bahtiyar G', 'orcid': None,...",2018,Curr Diab Rep,None,92,18,10,PURPOSE OF REVIEW: Glucagon-like peptide-1 rec...,13


## 6. Export results

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

deduplicated_path = OUTPUT_DIR / "diabetes_data_deduplicated.csv"
removed_path = OUTPUT_DIR / "diabetes_data_removed_duplicates.csv"
decisions_path = OUTPUT_DIR / "diabetes_data_cluster_decisions.csv"
scored_pairs_path = OUTPUT_DIR / "diabetes_data_scored_pairs.csv"

deduplicated_df.to_csv(deduplicated_path, index=False)
removed_duplicates_df.to_csv(removed_path, index=False)
decisions_df.to_csv(decisions_path, index=False)
scored_pairs_df.to_csv(scored_pairs_path, index=False)

## 7. Run summary

In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "input_records": len(papers),
            "candidate_pairs": len(scored_pairs_df),
            "predicted_duplicate_pairs": int(
                scored_pairs_df["duplicate_prediction"].sum()
            ),
            "threshold": THRESHOLD,
            "retention_strategy": RETENTION_STRATEGY,
            "records_kept": len(resolution.kept_records),
            "records_removed": len(resolution.removed_records),
        }
    ]
)

display(summary_df)

,input_records,candidate_pairs,predicted_duplicate_pairs,threshold,retention_strategy,records_kept,records_removed
0,1845,8487,2531,0.85,prefer_doi_abstract,593,1252
